In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Patparganj_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,355.0,57.0,194.0,130.0,222.0,277.0,103.0,32.0,98.0,155.0,346.0,293.0
1,2,362.0,148.0,108.0,153.0,191.0,180.0,129.0,56.0,81.0,183.0,337.0,286.0
2,3,350.0,157.0,112.0,219.0,268.0,NaN,108.0,68.0,73.0,172.0,407.0,271.0
3,4,405.0,262.0,163.0,196.0,282.0,253.0,50.0,53.0,65.0,239.0,403.0,176.0
4,5,370.0,139.0,124.0,172.0,337.0,292.0,72.0,51.0,51.0,157.0,382.0,160.0
5,6,343.0,140.0,117.0,188.0,NaN,267.0,58.0,59.0,81.0,149.0,380.0,195.0
6,7,376.0,159.0,156.0,175.0,NaN,243.0,45.0,50.0,59.0,134.0,399.0,217.0
7,8,389.0,145.0,141.0,175.0,283.0,272.0,53.0,47.0,103.0,167.0,402.0,328.0
8,9,395.0,130.0,127.0,196.0,218.0,228.0,92.0,58.0,118.0,191.0,371.0,176.0
9,10,305.0,332.0,209.0,251.0,208.0,209.0,136.0,65.0,105.0,146.0,351.0,247.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,355.000000,57.000000,194.0,130.0000,222.000000,277.0,103.000000,32.000000,98.000000,155.0,346.000000,293.000000
1,2,362.000000,148.000000,108.0,153.0000,191.000000,180.0,129.000000,56.000000,81.000000,183.0,337.000000,286.000000
2,3,350.000000,157.000000,112.0,219.0000,268.000000,173.6,108.000000,68.000000,73.000000,172.0,407.000000,271.000000
3,4,405.000000,262.000000,163.0,196.0000,282.000000,253.0,50.000000,53.000000,65.000000,239.0,403.000000,176.000000
4,5,370.000000,139.000000,124.0,172.0000,337.000000,292.0,72.000000,51.000000,51.000000,157.0,382.000000,160.000000
5,6,343.000000,140.000000,117.0,188.0000,211.333333,267.0,58.000000,59.000000,81.000000,149.0,380.000000,195.000000
6,7,376.000000,159.000000,156.0,175.0000,211.333333,243.0,45.000000,50.000000,59.000000,134.0,399.000000,217.000000
7,8,389.000000,145.000000,141.0,175.0000,283.000000,272.0,53.000000,47.000000,103.000000,167.0,402.000000,328.000000
8,9,395.000000,130.000000,127.0,196.0000,218.000000,228.0,92.000000,58.000000,118.000000,191.0,371.000000,176.000000
9,10,305.000000,332.000000,209.0,170.4375,208.000000,209.0,136.000000,65.000000,105.000000,146.0,351.000000,247.000000
